# ИИ-агент Travel Planner

Агент для планирования путешествий на базе LangGraph с внешними инструментами,
памятью и FastAPI-интерфейсом.

## Архитектура

Граф имеет 8 узлов и 2 условных ребра (3+ пути выполнения):

```mermaid
graph TD
    START --> classify_intent
    classify_intent -->|plan| select_destination
    classify_intent -->|refine| ask_missing_info
    classify_intent -->|recall| load_state
    select_destination --> check_weather_budget
    check_weather_budget -->|OK| build_itinerary
    check_weather_budget -->|bad| suggest_alternatives
    suggest_alternatives --> ask_missing_info
    build_itinerary --> save_state
    ask_missing_info --> END
    load_state --> END
    save_state --> END
```

## Узлы графа

| # | Узел | Модель | Описание |
|---|------|--------|----------|
| 1 | `classify_intent` | GigaChat-3-Pro | Классификация намерения: plan / refine / recall |
| 2 | `select_destination` | GigaChat-3-Pro | Выбор направления с учётом сезона и активности |
| 3 | `ask_missing_info` | GigaChat-3-Pro | Запрос недостающих параметров у пользователя |
| 4 | `load_state` | GigaChat-3-Pro | Восстановление состояния из JSON-файла |
| 5 | `check_weather_budget` | — | Параллельный вызов инструментов weather + costs |
| 6 | `suggest_alternatives` | GigaChat-3-Pro | Предложение альтернатив при плохой погоде/бюджете |
| 7 | `build_itinerary` | GigaChat3.5-Reasoning | Детальное составление маршрута с reasoning |
| 8 | `save_state` | — | Сериализация состояния в JSON |

## Внешние инструменты

| Инструмент | Источник | Описание |
|-----------|----------|----------|
| `get_weather_forecast` | Open-Meteo API | Прогноз погоды: температура, осадки, сезонные риски |
| `get_travel_costs` | Справочник цен | Оценка бюджета: перелёт, проживание, питание, развлечения |
| `get_exchange_rates` | exchangerate-api.com | Актуальные курсы валют (USD, EUR → RUB) |
| `get_accommodation_prices` | API жилья | Цены на жильё: хостелы, отели, апартаменты, виллы |
| `get_transport_options` | API транспорта | Варианты: авиа, ж/д, автобус, аренда авто |

## Память

- **Краткосрочная**: `InMemorySaver` (чекпоинтинг по thread_id)
- **Долгосрочная**: JSON-файлы в директории `states/`

## FastAPI-интерфейс

| Эндпоинт | Метод | Описание |
|----------|-------|----------|
| `/chat` | POST | Основной вход (синхронный) |
| `/chat/stream` | POST | Стриминг через SSE |
| `/state/{session_id}` | GET | Получение состояния сессии |
| `/reset/{session_id}` | POST | Очистка сессии |
| `/health` | GET | Health-check |


## Установка зависимостей

```bash
pip install -r requirements.txt
```

In [ ]:
import os
import json
import httpx
from typing import List, Annotated, Optional, Dict, Any, Literal
from typing_extensions import TypedDict
from datetime import datetime
from pathlib import Path

from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver
from openai import OpenAI

# --- Конфигурация ---
BASE_URL = "https://foundation-models.api.cloud.ru/v1"
DEFAULT_MODEL = "GigaChat/GigaChat-3-Pro"
REASONING_MODEL = "ai-sage/GigaChat3.5-432B-A28B-Reasoning"

# Ключ из переменной окружения
GIGACHAT_KEY = os.environ.get("GIGACHAT_KEY", "")

# OpenAI-совместимый клиент
client = OpenAI(
    api_key=GIGACHAT_KEY,
    base_url=BASE_URL,
) if GIGACHAT_KEY else None

# Проверка: клиент должен быть инициализирован
if client is None:
    raise RuntimeError(
        "GIGACHAT_KEY не установлен. Установите переменную окружения: "
        'export GIGACHAT_KEY="ваш_ключ"'
    )

print("Клиент инициализирован ✓")
print(f"Модель: {DEFAULT_MODEL}")


In [ ]:
# --- GraphState (TypedDict + Annotated) ---

class TravelState(TypedDict, total=False):
    user_id: str
    session_id: str
    messages: Annotated[List[dict], add_messages]  # reducer для истории
    intent: Literal["plan", "refine", "recall", "unknown"]
    destination: Optional[str]
    dates: Optional[str]
    budget: Optional[float]
    comfort_level: Optional[str]  # budget / standard / premium
    weather: Optional[str]
    costs: Optional[str]
    weather_ok: Optional[bool]
    budget_ok: Optional[bool]
    alternatives: Optional[str]
    itinerary: Optional[str]
    user_query: str
    agent_response: str
    asked_missing_info: bool  # флаг: уже задавали уточняющие вопросы


# --- Pydantic-схемы для структурного вывода ---

class IntentFormat(BaseModel):
    """Результат классификации намерения."""
    intent: Literal["plan", "refine", "recall", "unknown"] = Field(
        description="Намерение: plan/refine/recall/unknown"
    )
    missing_fields: List[str] = Field(
        default_factory=list,
        description="Список недостающих полей"
    )


class DestinationFormat(BaseModel):
    """Выбор направления."""
    destination: str = Field(description="Рекомендуемое направление")
    reason: str = Field(description="Причина выбора")
    temperature: str = Field(description="Ожидаемая температура")
    best_season: str = Field(description="Лучший сезон для посещения")


class MissingInfoFormat(BaseModel):
    """Уточняющий вопрос."""
    need_clarification: bool = Field(description="Нужно ли уточнение")
    missing_fields: List[str] = Field(description="Недостающие поля")
    query_to_user: str = Field(description="Уточняющий вопрос")


In [ ]:
# --- llm_call: parse-based (structured output) ---

def llm_call(
    dialog: list,
    structure: Optional[type] = None,
    model: str = DEFAULT_MODEL,
    temperature: float = 0.0,
    max_tokens: int = 4096,
) -> Any:
    """
    Универсальная функция вызова LLM.
    
    Args:
        dialog: Список сообщений для диалога
        structure: Pydantic-модель для структурного вывода
        model: Название модели
        temperature: Температура генерации
        max_tokens: Максимальное количество токенов
    
    Returns:
        Структурированные данные или текстовый ответ
    """
    request_params = {
        "model": model,
        "messages": dialog,
        "temperature": temperature,
        "max_tokens": max_tokens,
    }

    if structure is not None:
        # Parse-based подход: структурный вывод
        response = client.chat.completions.parse(
            model=model,
            messages=dialog,
            temperature=temperature,
            max_tokens=max_tokens,
            response_format=structure,
        )
        return response.choices[0].message.parsed.model_dump()
    else:
        # Текстовый ответ
        response = client.chat.completions.create(**request_params)
        return response.choices[0].message.content


In [17]:
# --- Tool 1: get_weather_forecast (Open-Meteo API) ---

def get_weather_forecast(location: str, dates: str) -> str:
    """
    Запрашивает прогноз погоды через Open-Meteo API.
    
    Args:
        location: Название города.
        dates: Диапазон дат (например, '2026-02-10,2026-02-17').
    
    Returns:
        Строка с информацией о погоде или сообщение об ошибке.
    """
    try:
        # Геокодинг
        geo_url = f"https://geocoding-api.open-meteo.com/v1/search?name={location}&count=1&language=ru"
        with httpx.Client(timeout=10.0) as http:
            geo_resp = http.get(geo_url)
            geo_data = geo_resp.json()
            
            if "results" not in geo_data or not geo_data["results"]:
                return f"Город '{location}' не найден в базе Open-Meteo."
            
            lat = geo_data["results"][0]["latitude"]
            lon = geo_data["results"][0]["longitude"]
            name = geo_data["results"][0]["name"]
            
            # Запрос погоды
            weather_url = (
                f"https://api.open-meteo.com/v1/forecast"
                f"?latitude={lat}&longitude={lon}"
                f"&daily=temperature_2m_max,temperature_2m_min,precipitation_sum,weathercode"
                f"&timezone=auto"
            )
            weather_resp = http.get(weather_url)
            weather_data = weather_resp.json()
            
            daily = weather_data.get("daily", {})
            temps_max = daily.get("temperature_2m_max", [])
            temps_min = daily.get("temperature_2m_min", [])
            precip = daily.get("precipitation_sum", [])
            codes = daily.get("weathercode", [])
            days = daily.get("time", [])
            
            if not temps_max:
                return "Сервис погоды временно недоступен, план составляется без учёта осадков."
            
            avg_temp = sum(temps_max) / len(temps_max)
            total_precip = sum(precip) if precip else 0
            
            def weather_desc(code):
                descriptions = {
                    0: "Ясно", 1: "Преимущественно ясно", 2: "Переменная облачность",
                    3: "Пасмурно", 45: "Туман", 48: "Изморозь",
                    51: "Лёгкая морось", 53: "Морось", 55: "Сильная морось",
                    61: "Небольшой дождь", 63: "Дождь", 65: "Сильный дождь",
                    71: "Небольшой снег", 73: "Снег", 75: "Сильный снег",
                    80: "Небольшой ливень", 81: "Ливень", 82: "Сильный ливень",
                    95: "Гроза", 96: "Гроза с градом", 99: "Сильная гроза с градом"
                }
                return descriptions.get(code, "Неизвестно")
            
            # Определяем сезонные риски
            seasonal_risks = []
            if avg_temp < 10:
                seasonal_risks.append("холодная погода, нужна тёплая одежда")
            elif avg_temp < 20:
                seasonal_risks.append("прохладная погода")
            elif avg_temp > 30:
                seasonal_risks.append("жара, нужна защита от солнца")
            if total_precip > 30:
                seasonal_risks.append("вероятны дожди")
            if any(c in [95, 96, 99] for c in codes):
                seasonal_risks.append("грозы возможны")
            
            result = (
                f"**Погода в {name}** ({', '.join(days[:3])}...):\n"
                f"- Средняя макс. температура: {avg_temp:.1f}°C\n"
                f"- Мин. температура: {min(temps_min):.1f}°C\n"
                f"- Осадки за период: {total_precip:.1f} мм\n"
                f"- Преобладающая погода: {weather_desc(codes[0] if codes else 0)}\n"
            )
            if seasonal_risks:
                result += f"- Риски: {', '.join(seasonal_risks)}\n"
            
            return result
    
    except httpx.TimeoutException:
        return "Сервис погоды временно недоступен (таймаут), план составляется без учёта осадков."
    except Exception as e:
        return f"Ошибка при запросе погоды: {e}. План составляется без учёта осадков."


# Тест инструмента
# print(get_weather_forecast("Amman", "2026-02-10,2026-02-17"))

In [18]:
# --- Tool 2: get_travel_costs (статичный справочник) ---

# Справочник средних цен (в USD на человека в день)
COSTS_DB = {
    "Turkey": {
        "flight": {"budget": 350, "standard": 600, "premium": 1200},
        "hotel_per_day": {"budget": 40, "standard": 80, "premium": 200},
        "food_per_day": {"budget": 15, "standard": 30, "premium": 70},
        "activities_per_day": {"budget": 10, "standard": 25, "premium": 60},
        "currency": "TRY (1 USD ≈ 35 TRY)",
        "note": "Сезон: май-октябрь"
    },
    "Jordan": {
        "flight": {"budget": 450, "standard": 700, "premium": 1400},
        "hotel_per_day": {"budget": 50, "standard": 100, "premium": 250},
        "food_per_day": {"budget": 20, "standard": 40, "premium": 80},
        "activities_per_day": {"budget": 30, "standard": 60, "premium": 120},
        "currency": "JOD (1 USD ≈ 0.71 JOD)",
        "note": "Сезон: март-май, сентябрь-ноябрь"
    },
    "Thailand": {
        "flight": {"budget": 500, "standard": 800, "premium": 1800},
        "hotel_per_day": {"budget": 25, "standard": 60, "premium": 180},
        "food_per_day": {"budget": 10, "standard": 25, "premium": 60},
        "activities_per_day": {"budget": 15, "standard": 40, "premium": 100},
        "currency": "THB (1 USD ≈ 35 THB)",
        "note": "Сезон: ноябрь-март"
    },
    "Egypt": {
        "flight": {"budget": 400, "standard": 650, "premium": 1300},
        "hotel_per_day": {"budget": 35, "standard": 70, "premium": 180},
        "food_per_day": {"budget": 15, "standard": 30, "premium": 65},
        "activities_per_day": {"budget": 20, "standard": 45, "premium": 100},
        "currency": "EGP (1 USD ≈ 50 EGP)",
        "note": "Сезон: октябрь-апрель"
    },
    "UAE": {
        "flight": {"budget": 450, "standard": 750, "premium": 1600},
        "hotel_per_day": {"budget": 60, "standard": 120, "premium": 350},
        "food_per_day": {"budget": 25, "standard": 50, "premium": 120},
        "activities_per_day": {"budget": 20, "standard": 50, "premium": 150},
        "currency": "AED (1 USD ≈ 3.67 AED)",
        "note": "Сезон: ноябрь-март"
    },
    "Georgia": {
        "flight": {"budget": 250, "standard": 450, "premium": 900},
        "hotel_per_day": {"budget": 30, "standard": 60, "premium": 150},
        "food_per_day": {"budget": 12, "standard": 25, "premium": 50},
        "activities_per_day": {"budget": 10, "standard": 20, "premium": 50},
        "currency": "GEL (1 USD ≈ 2.7 GEL)",
        "note": "Сезон: круглый год"
    },
    "Maldives": {
        "flight": {"budget": 600, "standard": 1000, "premium": 2500},
        "hotel_per_day": {"budget": 100, "standard": 300, "premium": 800},
        "food_per_day": {"budget": 40, "standard": 80, "premium": 200},
        "activities_per_day": {"budget": 30, "standard": 80, "premium": 200},
        "currency": "MVR (1 USD ≈ 15.4 MVR)",
        "note": "Сезон: ноябрь-апрель"
    },
    "Morocco": {
        "flight": {"budget": 400, "standard": 650, "premium": 1300},
        "hotel_per_day": {"budget": 30, "standard": 70, "premium": 200},
        "food_per_day": {"budget": 15, "standard": 35, "premium": 70},
        "activities_per_day": {"budget": 15, "standard": 35, "premium": 80},
        "currency": "MAD (1 USD ≈ 10 MAD)",
        "note": "Сезон: март-май, сентябрь-ноябрь"
    }
}


def get_travel_costs(location: str, days: int, comfort_level: str) -> str:
    """
    Возвращает оценку стоимости поездки на основе справочника.
    
    Args:
        location: Название страны/региона.
        days: Количество дней.
        comfort_level: Уровень комфорта (budget / standard / premium).
    
    Returns:
        Строка с оценкой бюджета или сообщение об ошибке.
    """
    try:
        # Нормализация названия
        location_lower = location.lower()
        matched_key = None
        for key in COSTS_DB:
            if key.lower() in location_lower or location_lower in key.lower():
                matched_key = key
                break
        
        if matched_key is None:
            return (
                f"Нет данных о стоимости для '{location}'. "
                f"Доступные направления: {', '.join(COSTS_DB.keys())}. "
                "Оценка бюджета строится на основе похожих направлений."
            )
        
        costs = COSTS_DB[matched_key]
        comfort = comfort_level if comfort_level in costs["flight"] else "standard"
        
        flight = costs["flight"][comfort]
        hotel = costs["hotel_per_day"][comfort] * days
        food = costs["food_per_day"][comfort] * days
        activities = costs["activities_per_day"][comfort] * days
        total = flight + hotel + food + activities
        
        result = (
            f"**Стоимость поездки в {matched_key}** ({days} дн., {comfort}):\n"
            f"- Перелёт: ~${flight}\n"
            f"- Проживание: ~${hotel} ({costs['hotel_per_day'][comfort]}/день)\n"
            f"- Питание: ~${food} ({costs['food_per_day'][comfort]}/день)\n"
            f"- Развлечения: ~${activities} ({costs['activities_per_day'][comfort]}/день)\n"
            f"- **Итого: ~${total}**\n"
            f"- Валюта: {costs['currency']}\n"
            f"- Сезон: {costs['note']}"
        )
        return result
    
    except Exception as e:
        return f"Ошибка при расчёте стоимости: {e}. Оценка бюджета приблизительная."


# Тест инструмента
# print(get_travel_costs("Jordan", 7, "standard"))

In [19]:
# --- Tool 3: get_exchange_rates (внешний API — курсы валют) ---

def get_exchange_rates(from_currency: str = "USD", to_currency: str = "RUB") -> str:
    """
    Запрашивает актуальный курс валют через бесплатный API.
    
    Args:
        from_currency: Код валюты-источника (USD, EUR, GBP)
        to_currency: Код валюты-получателя (RUB, USD, EUR)
    
    Returns:
        Строка с курсом или сообщение об ошибке.
    """
    try:
        url = f"https://api.exchangerate-api.com/v4/latest/{from_currency}"
        with httpx.Client(timeout=10.0) as http:
            resp = http.get(url)
            resp.raise_for_status()
            data = resp.json()
            
            rate = data["rates"].get(to_currency)
            if rate is None:
                return f"Валюта {to_currency} не найдена в ответе API."
            
            return (
                f"Курс валют: 1 {from_currency} = {rate} {to_currency}\n"
                f"Дата обновления: {data.get('time_last_update_utc', 'N/A')}"
            )
    
    except httpx.TimeoutException:
        return "Сервис курсов валют временно недоступен (таймаут)."
    except Exception as e:
        return f"Ошибка при запросе курса валют: {e}."


# Тест инструмента
# print(get_exchange_rates("USD", "RUB"))

In [20]:
# --- Tool 4: get_accommodation_prices (API поиска жилья) ---

def get_accommodation_prices(location, check_in, check_out, guests=1):
    """
    Запрашивает цены на жилье.
    В реальном проекте: Booking.com API, Airbnb API.
    """
    try:
        accommodation_types = {
            "hostel": {"price_per_night": 15, "description": "Хостел (общая комната)"},
            "budget_hotel": {"price_per_night": 40, "description": "Бюджетный отель"},
            "standard_hotel": {"price_per_night": 80, "description": "Стандартный отель"},
            "premium_hotel": {"price_per_night": 180, "description": "Премиум отель"},
            "apartment": {"price_per_night": 60, "description": "Апартаменты/квартира"},
            "villa": {"price_per_night": 250, "description": "Вилла/дом"},
        }
        
        from datetime import datetime
        start = datetime.strptime(check_in, "%Y-%m-%d")
        end = datetime.strptime(check_out, "%Y-%m-%d")
        nights = (end - start).days
        
        result = f"**Доступное жилье в {location}** ({nights} ночей, {guests} гост.):\n"
        
        for key, info in accommodation_types.items():
            total = info["price_per_night"] * nights
            result += f"- {info['description']}: ${info['price_per_night']}/ночь (всего: ${total})\n"
        
        return result
    
    except Exception as e:
        return f"Ошибка при запросе цен на жилье: {e}."

# Тест инструмента
# print(get_accommodation_prices("Istanbul", "2026-02-10", "2026-02-17"))

In [21]:
# --- Tool 5: get_transport_options (API поиска транспорта) ---

def get_transport_options(from_city, to_city, travel_date, passengers=1):
    """
    Запрашивает варианты транспорта.
    В реальном проекте: Aviasales API, RZD API, Rentalcars API.
    """
    try:
        transport_types = {
            "flight": {
                "price_per_person": 150,
                "duration": "2-4 часа",
                "description": "Авиабилет",
                "booking_url": "https://www.aviasales.ru"
            },
            "train": {
                "price_per_person": 80,
                "duration": "6-12 часов",
                "description": "Ж/д билет",
                "booking_url": "https://www.rzd.ru"
            },
            "bus": {
                "price_per_person": 40,
                "duration": "8-15 часов",
                "description": "Автобус",
                "booking_url": "https://www.pulscen.ru"
            },
            "car_rental": {
                "price_per_day": 50,
                "duration": "гибко",
                "description": "Аренда авто",
                "booking_url": "https://www.rentalcars.com"
            },
        }
        
        result = f"**Варианты транспорта {from_city} -> {to_city}** ({travel_date}, {passengers} пасс.):\n"
        
        for key, info in transport_types.items():
            if key == "car_rental":
                price_info = f"${info['price_per_day']}/день"
            else:
                price_info = f"${info['price_per_person']}/чел"
            
            result += (
                f"- {info['description']}: {price_info}, "
                f"в пути: {info['duration']}\n"
                f"  Бронирование: {info['booking_url']}\n"
            )
        
        return result
    
    except Exception as e:
        return f"Ошибка при запросе вариантов транспорта: {e}."

# Тест инструмента
# print(get_transport_options("Moscow", "Sochi", "2026-02-10"))

In [23]:
# --- Node: select_destination ---
def select_destination(state: TravelState) -> dict:
    from pydantic import BaseModel
    class DestinationFormat(BaseModel):
        destination: str
        reason: str
        temperature: str
        best_season: str
    user_query = state.get("user_query", "")
    
    # Добавляем контекст: указываем доступные направления и их сезоны
    context = (
        "Доступные направления и их сезоны:\n"
        "- Горнолыжные (Россия, Грузия): декабрь-март\n"
        "- Пляжные (Египет, Турция): март-октябрь\n"
        "- Тропики (Таиланд, Мальдивы): ноябрь-март\n"
        "- Ближний Восток (Иордания, ОАЭ): март-май, сентябрь-ноябрь\n"
        "- Кавказ (Грузия): круглый год\n\n"
        "Выбери направление, которое соответствует запросу пользователя по сезону и активности."
    )
    
    dialog = [
        {"role": "system", "content": "Подбери направление. Ответь 1 предложением." + context},
        {"role": "user", "content": user_query},
    ]
    result = llm_call(dialog, structure=DestinationFormat)
    return {"destination": result["destination"],
            "messages": [{"role": "assistant", "content": f"Рекомендую: {result['destination']}"}]}


In [24]:
# --- Роутинг (conditional edges) ---

def route_intent(state: TravelState) -> str:
    """
    Роутер после classify_intent.
    Возвращает строку для выбора узла.
    """
    intent = state.get("intent", "unknown")
    if isinstance(intent, dict):
        intent = intent.get("intent", "unknown")
    
    if intent == "recall":
        return "load_state"
    elif intent == "plan":
        return "select_destination"
    elif intent == "refine":
        return "ask_missing_info"
    else:
        return "ask_missing_info"


def route_check(state: TravelState) -> str:
    """
    Роутер после check_weather_budget.
    build_itinerary если всё OK, suggest_alternatives если нет.
    """
    weather_ok = state.get("weather_ok", True)
    budget_ok = state.get("budget_ok", True)
    
    if not weather_ok or not budget_ok:
        return "suggest_alternatives"
    return "build_itinerary"

In [25]:
# --- Компиляция графа (StateGraph + InMemorySaver) ---

# Создаём граф
graph_builder = StateGraph(TravelState)

# Добавляем узлы
graph_builder.add_node("classify_intent", classify_intent)
graph_builder.add_node("select_destination", select_destination)
graph_builder.add_node("ask_missing_info", ask_missing_info)
graph_builder.add_node("load_state", load_state)
graph_builder.add_node("check_weather_budget", check_weather_budget)
graph_builder.add_node("suggest_alternatives", suggest_alternatives)
graph_builder.add_node("build_itinerary", build_itinerary)
graph_builder.add_node("save_state", save_state)

# Добавляем рёбра
graph_builder.add_edge(START, "classify_intent")

# Условные рёбра после classify_intent (AIA_06 line 626)
graph_builder.add_conditional_edges(
    "classify_intent",
    route_intent,
    {
        "select_destination": "select_destination",
        "ask_missing_info": "ask_missing_info",
        "load_state": "load_state",
    },
)

# Прямые рёбра
graph_builder.add_edge("select_destination", "check_weather_budget")
graph_builder.add_edge("ask_missing_info", END)
graph_builder.add_edge("load_state", END)

# Условные рёбра после check_weather_budget
graph_builder.add_conditional_edges(
    "check_weather_budget",
    route_check,
    {
        "build_itinerary": "build_itinerary",
        "suggest_alternatives": "suggest_alternatives",
    },
)

# После suggest_alternatives → ask_missing_info (для подтверждения)
graph_builder.add_edge("suggest_alternatives", "ask_missing_info")

# После build_itinerary → save_state → END
graph_builder.add_edge("build_itinerary", "save_state")
graph_builder.add_edge("save_state", END)

# Компилируем с InMemorySaver (AIA_04 line 76)
checkpointer = InMemorySaver()
graph = graph_builder.compile(checkpointer=checkpointer)

print("Граф скомпилирован успешно!")
print(f"Узлов: {len(graph_builder.nodes)}")
print(f"Рёбер: {len(graph_builder.edges)}")

Граф скомпилирован успешно!
Узлов: 8
Рёбер: 7


In [ ]:
# --- Визуализация графа (как в AIA_06) ---

from IPython.display import display, Markdown, Image

# Способ 1: Mermaid-диаграмма (текстовая)
print("=== Граф агента Travel Planner ===")
print()
display(Markdown(graph.get_graph().draw_mermaid()))

print()
print("=== PNG-изображение графа ===")
try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"Не удалось отобразить PNG: {e}")


In [26]:
# --- Демонстрация: Шаг 1 ---
# Пользователь: «Хочу поехать в тепло в феврале, бюджет 70 000 рублей»
# Агент → classify_intent → select_destination → check_weather_budget

from pprint import pprint

user_query_1 = "Хочу поехать в тепло в феврале, бюджет 70 000 рублей"

initial_state = {
    "user_id": "troshin_vlad",
    "session_id": "session_1",
    "user_query": user_query_1,
    "budget": 70000,
    "dates": "2026-02-10,2026-02-17",
    "comfort_level": "standard",
}

config = {"configurable": {"thread_id": "troshin_vlad_session_1"}}

print("=" * 60)
print("ШАГ 1: classify_intent → select_destination → check_weather_budget")
print("=" * 60)
print(f"Запрос: {user_query_1}")
print()

result_1 = graph.invoke(initial_state, config)

print()
print("-" * 40)
print("РЕЗУЛЬТАТ ШАГА 1:")
print("-" * 40)
print(f"Направление: {result_1.get('destination', 'N/A')}")
print(f"Погода OK: {result_1.get('weather_ok', 'N/A')}")
print(f"Бюджет OK: {result_1.get('budget_ok', 'N/A')}")
print()
print("Погода:")
print(result_1.get('weather', 'N/A')[:500])
print()
print("Стоимость:")
print(result_1.get('costs', 'N/A')[:500])

ШАГ 1: classify_intent → select_destination → check_weather_budget
Запрос: Хочу поехать в тепло в феврале, бюджет 70 000 рублей

[classify_intent]
  intent=plan, missing=['destination', 'dates']
[select_destination]
  destination=Thailand
[check_weather_budget]
  weather: Город 'Thailand' не найден в базе Open-Meteo....
  costs: **Стоимость поездки в Thailand** (7 дн., standard):
- Перелёт: ~$800
- Проживани...
[build_itinerary]


TypeError: 'NoneType' object is not subscriptable

In [ ]:
# --- Демонстрация: Шаг 2 ---
# Если погода плохая → suggest_alternatives → Иордания
# Пользователь: «Да, подтверждаю»

print("=" * 60)
print("ШАГ 2: suggest_alternatives (если погода/бюджет не OK)")
print("=" * 60)

# Проверяем, нужно ли предлагать альтернативы
weather_ok = result_1.get("weather_ok", True)
budget_ok = result_1.get("budget_ok", True)

if not weather_ok or not budget_ok:
    print(f"Погода OK: {weather_ok}, Бюджет OK: {budget_ok}")
    print("Альтернативы предлагаются...")
    
    # Продолжаем с тем же состоянием
    result_2 = graph.invoke(result_1, config)
    
    print()
    print("Альтернативы:")
    print(result_2.get("alternatives", "N/A")[:800])
else:
    print(f"Погода OK: {weather_ok}, Бюджет OK: {budget_ok}")
    print("Альтернативы не нужны, переходим к составлению маршрута.")
    result_2 = result_1

In [ ]:
# --- Демонстрация: Шаг 3 ---
# build_itinerary → save_state → финальный результат

print("=" * 60)
print("ШАГ 3: build_itinerary → save_state")
print("=" * 60)

# Если на шаге 2 были альтернативы, продолжаем с ними
if "alternatives" in result_2:
    print("Альтернативы предложены, составляем маршрут...")
else:
    print("Продолжаем с текущим направлением...")

result_3 = graph.invoke(result_2, config)

print()
print("=" * 60)
print("ФИНАЛЬНЫЙ РЕЗУЛЬТАТ")
print("=" * 60)
print()
itinerary = result_3.get("itinerary", "")
if itinerary:
    print(itinerary[:2000])
    if len(itinerary) > 2000:
        print(f"\n... (ещё {len(itinerary) - 2000} символов)")
else:
    print("Маршрут не сгенерирован.")

print()
print("Сохранённые файлы:")
import os
states_dir = Path("states")
if states_dir.exists():
    for f in states_dir.iterdir():
        print(f"  {f}")

In [30]:
# --- Node: build_itinerary ---
def build_itinerary(state: TravelState) -> dict:
    destination = state.get("destination", "")
    weather = state.get("weather", "")
    costs = state.get("costs", "")
    budget = state.get("budget", 70000)
    dates = state.get("dates", "")
    comfort = state.get("comfort_level", "standard")
    user_query = state.get("user_query", "")
    dialog = [
        {"role": "system", "content": "Составь план. Отвечай кратко, по пунктам. Используй точные даты из запроса."},
        {"role": "user", "content": f"Направление: {destination}. Даты: {dates}. Бюджет: {budget} руб. Уровень комфорта: {comfort}. Запрос: {user_query}"},
    ]
    result = llm_call(dialog, model=REASONING_MODEL)
    return {"itinerary": result,
            "messages": [{"role": "assistant", "content": result}]}


Overwriting app.py
